# Precise IHC HER2 — Point-Detection Evaluation (YOLOv11)

HER2 IHC 세포 검출/분류 모델의 **point-detection 기반** 정량 평가 노트북.

**Metrics Philosophy**
- 고정 bbox 기반 point detection network이므로 box IoU 대신 **중심점 거리 매칭**을 사용.
- **Detection**(탐지)과 **Classification**(분류)를 분리해서 측정.

**Key Metrics**
- **DQ** (Detection Quality): class-agnostic 탐지 F1
- **CQ** (Classification Quality): 매칭된 점의 분류 정확도
- **PQ** (Panoptic Quality) = DQ × CQ
- **MLE** (Mean Localization Error): 매칭 쌍의 평균 유클리드 거리 (px)
- **FROC**: Sensitivity vs Avg FP/image

**Analysis Contents**
1. Data & Model Loading
2. Table 1 — Per-class Detection + Classification
3. Table 2 — Point Detection Summary (DQ/CQ/PQ/MLE/FROC)
4. Figure 1 — Confusion Matrix
5. Figure 2 — Per-class P/R/F1 Bar Chart
6. Figure 3 — Precision-Recall Curves
7. Figure 4 — FROC Curve & Confidence Sweep
8. Figure 5 — Qualitative GT vs Prediction
9. Figure 6 — Distance Threshold Sensitivity
10. Figure 7 — TP/FP/FN Error Analysis
11. Figure 8 — Per-image GT vs Pred Count
12. CSV / LaTeX Export


In [ ]:
import copy, csv, os, warnings, glob, json, random, importlib, struct, zlib, io
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
from nets import nn
from utils import util
importlib.reload(util)  # utils.py 수정사항 반영 (class-agnostic NMS 등)
import pandas as pd
from utils.dataset import Dataset
from PIL import Image, PngImagePlugin
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from scipy.optimize import linear_sum_assignment

# ── Plot style for publication ──────────────────────────────────
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
})
warnings.filterwarnings('ignore')

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)


## 1. Data Loading & Model Checkpoint

In [ ]:
input_size = 512

# ── 경로 설정 (IHC HER2) ────────────────────────────────────────
label_dir = '../../data/precise_BC_cell_scoring/her2/labels/'
image_dir = '../../data/precise_BC_cell_scoring/her2/patch_images/'
label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

# ── 5 클래스 정의 (HER2 IHC scoring) ────────────────────────────
class_names_her2 = {
    0: "class0",   # 0+
    1: "class1",   # 1+
    2: "class2",   # 2+
    3: "class3",   # 3+
    4: "other"
}
num_classes_her2 = len(class_names_her2)
params['names'] = class_names_her2

# figure 용 짧은 이름
short_names = {
    0: '0+', 1: '1+', 2: '2+', 3: '3+', 4: 'other'
}

# ── 라벨 로드 (JSON: [{class_id, cx, cy, w, h}, ...], normalized) ──
image_filenames, labels = [], []
print("Loading labels...")
for label_file in tqdm(label_files):
    base_name = os.path.splitext(os.path.basename(label_file))[0]
    img_path = os.path.join(image_dir, base_name + '.png')
    if not os.path.exists(img_path):
        continue
    with open(label_file) as f:
        data_json = json.load(f)
    boxes = []
    for ann in data_json:
        cls_id = int(ann['class_id'])
        cx, cy, w, h = ann['cx'], ann['cy'], ann['w'], ann['h']
        cx_px = cx * input_size
        cy_px = cy * input_size
        w_px = w * input_size
        h_px = h * input_size
        x_left = cx_px - w_px / 2.0
        y_top  = cy_px - h_px / 2.0
        boxes.append([cls_id, float(y_top), float(x_left), float(h_px), float(w_px)])
    if len(boxes) > 0:
        image_filenames.append(img_path)
        labels.append(boxes)
print(f"✅ {len(image_filenames)} images with labels loaded")

# ── 이미지 로드 (ICC 프로파일 스킵) ─────────────────────────────
def read_png_skip_icc(path):
    """iCCP/텍스트 청크를 건너뛰고 PNG를 읽는다."""
    with open(path, 'rb') as f:
        signature = f.read(8)
        chunks = []
        while True:
            raw = f.read(8)
            if len(raw) < 8:
                break
            length, ctype = struct.unpack('>I4s', raw)
            data_b = f.read(length)
            crc = f.read(4)
            if ctype in (b'iCCP', b'tEXt', b'iTXt', b'zTXt'):
                continue
            chunks.append(raw + data_b + crc)
    buf = io.BytesIO(signature + b''.join(chunks))
    return Image.open(buf).convert('RGB')

print("Loading images...")
images = []
valid_labels = []
skip_count = 0
for img_path, lbl in tqdm(list(zip(image_filenames, labels)), total=len(image_filenames)):
    try:
        img = np.array(read_png_skip_icc(img_path))
    except Exception:
        skip_count += 1
        continue
    images.append(img)
    valid_labels.append(lbl)
labels = valid_labels
print(f"✅ {len(images)} images loaded  |  skipped: {skip_count}  |  shape: {images[0].shape}")


In [ ]:
# ── Dataset & collate (train 노트북과 동일, augment=False) ──────
def collate_fn1(batch):
    samples, cls, box, indices = zip(*batch)
    cls = torch.cat(cls, dim=0)
    box = torch.cat(box, dim=0)
    new_indices = list(indices)
    for i in range(len(indices)):
        new_indices[i] += i
    indices = torch.cat(new_indices, dim=0)
    targets = {'cls': cls, 'box': box, 'idx': indices}
    return torch.stack(samples, dim=0), targets


class custom_dataset(data.Dataset):
    def __init__(self, images, params, augment, labels):
        self.params = params
        self.augment = augment
        self.images  = images
        self.labels  = labels
        self.base_size = 512
        self.current_scale = 512
        self.n = len(self.images)
        self.indices = list(range(self.n))

    def __len__(self):
        return len(self.indices)

    def set_scale(self, scale):
        self.current_scale = scale

    def crop_padding_image(self, image):
        image = image.copy()
        h, w = image.shape[:2]
        S = self.base_size
        if min(h, w) > S:
            row_off = random.randint(0, h - S)
            col_off = random.randint(0, w - S)
            image = image[row_off:row_off + S, col_off:col_off + S]
        else:
            row_off, col_off = 0, 0
            pad = np.ones((S, S, 3), dtype=np.uint8) * 255
            pad[:min(h, S), :min(w, S)] = image[:min(h, S), :min(w, S)]
            image = pad
        return image, row_off, col_off

    def __getitem__(self, index):
        index = self.indices[index]
        original_image = self.images[index].copy()
        raw_labels = self.labels[index]
        S = self.base_size

        crop_boxes = []
        while len(crop_boxes) < 1:
            image, row_off, col_off = self.crop_padding_image(original_image)
            crop_boxes = []
            for lbl in raw_labels:
                cls_id, y_top, x_left, h, w = lbl
                cy_center = y_top  + h / 2.0
                cx_center = x_left + w / 2.0
                if not (row_off <= cy_center <= row_off + S and
                        col_off <= cx_center <= col_off + S):
                    continue
                new_y = np.clip(y_top  - row_off, 0, S)
                new_x = np.clip(x_left - col_off, 0, S)
                new_h = np.clip(h, 0, S - new_y)
                new_w = np.clip(w, 0, S - new_x)
                if new_h < 1 or new_w < 1:
                    continue
                cx_norm = (new_x + new_w / 2.0) / S
                cy_norm = (new_y + new_h / 2.0) / S
                w_norm  = new_w / S
                h_norm  = new_h / S
                crop_boxes.append([int(cls_id), cx_norm, cy_norm, w_norm, h_norm])

        boxes = np.array(crop_boxes, dtype=np.float32)
        size = self.current_scale
        if size != S:
            image = cv2.resize(image, (size, size))
        image = image.transpose((2, 0, 1))
        cls_arr = boxes[:, 0].astype(np.float32)
        box_arr = boxes[:, 1:5].astype(np.float32)
        nl = len(box_arr)
        return (torch.from_numpy(image),
                torch.from_numpy(cls_arr),
                torch.from_numpy(box_arr),
                torch.zeros(nl))


# ── Train 노트북과 동일한 split 사용 (val split 평가) ───────────
patch_train, patch_test, label_train, label_test = train_test_split(
    images, labels, test_size=0.2, random_state=242, shuffle=True
)

val_dataset = custom_dataset(patch_test, params, augment=False, labels=label_test)
val_loader  = torch.utils.data.DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    num_workers=4, collate_fn=collate_fn1, drop_last=False
)
print(f"✅ Val set: {len(val_dataset)} images, {len(val_loader)} batches")


In [ ]:
# ── 모델 로드 ──────────────────────────────────────────────────
save_dir  = '../../model/precise_BC_cell_scoring/her2_yolov11/'
ckpt_path = os.path.join(save_dir, 'best_model.pt')

model = nn.yolo_v11_m(num_classes_her2).to(device)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Best model loaded from: {ckpt_path}")
print(f"   Epoch           : {checkpoint.get('epoch', 'N/A')}")
try:
    print(f"   Overall Recall  : {checkpoint.get('overall_recall', 0):.4f}")
    print(f"   Macro F1        : {checkpoint.get('macro_f1', 0):.4f}")
except Exception:
    pass


## 2. Full Evaluation — Collect All Predictions & Matches

In [ ]:
def compute_iou_matrix(boxes1, boxes2):
    """IoU matrix between two sets of xyxy boxes. boxes1: (N,4), boxes2: (M,4)."""
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)
    b1 = np.asarray(boxes1, dtype=np.float32)
    b2 = np.asarray(boxes2, dtype=np.float32)
    x1 = np.maximum(b1[:, None, 0], b2[None, :, 0])
    y1 = np.maximum(b1[:, None, 1], b2[None, :, 1])
    x2 = np.minimum(b1[:, None, 2], b2[None, :, 2])
    y2 = np.minimum(b1[:, None, 3], b2[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a1 = (b1[:, 2] - b1[:, 0]) * (b1[:, 3] - b1[:, 1])
    a2 = (b2[:, 2] - b2[:, 0]) * (b2[:, 3] - b2[:, 1])
    union = a1[:, None] + a2[None, :] - inter + 1e-9
    return inter / union


def full_evaluation(model, loader, device, params,
                    conf_threshold=0.1, nms_iou_threshold=0.3,
                    iou_match_threshold=0.3):
    """
    IoU 기반 detection 평가.
    - NMS: class-agnostic (다른 클래스 간 중복 예측 제거).
    - Matching: Hungarian on IoU (IoU >= iou_match_threshold인 쌍만 매칭).
    - Detection(탐지)과 Classification(분류)를 분리해서 측정.
    """
    model.eval()
    num_cls = len(params['names'])
    img_size = 512

    int_total_gt      = 0
    int_total_pred    = 0
    int_total_matched = 0
    int_correct_cls   = 0

    class_tp = np.zeros(num_cls)
    class_fp = np.zeros(num_cls)
    class_fn = np.zeros(num_cls)
    class_gt_count = np.zeros(num_cls)

    confusion = np.zeros((num_cls, num_cls), dtype=np.int64)
    list_matched_ious      = []
    list_matched_distances = []

    list_all_detections = []
    np_all_gt_counts    = np.zeros(num_cls)

    list_per_image_gt   = []
    list_per_image_pred = []
    list_per_image_tp   = []
    list_per_image_fp   = []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc='Evaluating'):
            imgs = imgs.to(device).float() / 255.
            with torch.amp.autocast('cuda'):
                pred = model(imgs)
            nms_results = util.non_max_suppression(
                pred, confidence_threshold=conf_threshold,
                iou_threshold=nms_iou_threshold,
                class_agnostic=True)  # 🎯 모든 클래스 통합 NMS

            for i in range(len(imgs)):
                idx_mask = targets['idx'] == i
                if not idx_mask.any():
                    list_per_image_gt.append(np.zeros(num_cls))
                    list_per_image_pred.append(np.zeros(num_cls))
                    list_per_image_tp.append(0)
                    list_per_image_fp.append(0)
                    continue

                gt_cls = targets['cls'][idx_mask].cpu().numpy()
                gt_box_nrm = targets['box'][idx_mask].cpu().numpy()  # cx,cy,w,h normalized
                # → xyxy pixel
                gt_cx = gt_box_nrm[:, 0] * img_size
                gt_cy = gt_box_nrm[:, 1] * img_size
                gt_w  = gt_box_nrm[:, 2] * img_size
                gt_h  = gt_box_nrm[:, 3] * img_size
                gt_xyxy = np.stack([gt_cx - gt_w/2, gt_cy - gt_h/2,
                                    gt_cx + gt_w/2, gt_cy + gt_h/2], axis=1)
                gt_centers = np.stack([gt_cx, gt_cy], axis=1)

                gt_cnt = np.zeros(num_cls)
                for c in gt_cls:
                    gt_cnt[int(c)] += 1
                    class_gt_count[int(c)] += 1
                    np_all_gt_counts[int(c)] += 1
                list_per_image_gt.append(gt_cnt)
                int_total_gt += len(gt_cls)

                pred_cnt = np.zeros(num_cls)
                img_tp, img_fp = 0, 0

                if len(nms_results) > i and len(nms_results[i]) > 0:
                    det = nms_results[i].cpu().numpy()
                    pred_boxes   = det[:, :4]  # xyxy
                    pred_confs   = det[:, 4]
                    pred_classes = det[:, 5].astype(int)
                    int_total_pred += len(pred_classes)

                    pred_centers = np.stack([
                        (pred_boxes[:, 0] + pred_boxes[:, 2]) / 2,
                        (pred_boxes[:, 1] + pred_boxes[:, 3]) / 2], axis=1)

                    iou_mat = compute_iou_matrix(gt_xyxy, pred_boxes)
                    # Hungarian maximizes IoU → minimize (-IoU)
                    cost = -iou_mat

                    matched_gt, matched_pred = set(), set()
                    if cost.size > 0:
                        gi, pi = linear_sum_assignment(cost)
                        for g, p in zip(gi, pi):
                            if iou_mat[g, p] >= iou_match_threshold:
                                matched_gt.add(g)
                                matched_pred.add(p)
                                int_total_matched += 1
                                list_matched_ious.append(iou_mat[g, p])
                                dx = gt_centers[g, 0] - pred_centers[p, 0]
                                dy = gt_centers[g, 1] - pred_centers[p, 1]
                                list_matched_distances.append(float(np.sqrt(dx*dx + dy*dy)))
                                img_tp += 1

                                gc = int(gt_cls[g])
                                pc = pred_classes[p]
                                confusion[gc, pc] += 1

                                if gc == pc:
                                    int_correct_cls += 1
                                    class_tp[gc] += 1
                                    list_all_detections.append((pred_confs[p], pc, True, gc))
                                else:
                                    class_fn[gc] += 1
                                    class_fp[pc] += 1
                                    list_all_detections.append((pred_confs[p], pc, False, gc))

                    for g in range(len(gt_cls)):
                        if g not in matched_gt:
                            class_fn[int(gt_cls[g])] += 1

                    for p in range(len(pred_classes)):
                        if p not in matched_pred:
                            class_fp[pred_classes[p]] += 1
                            img_fp += 1
                            list_all_detections.append((pred_confs[p], pred_classes[p], False, -1))

                    for c in pred_classes:
                        pred_cnt[c] += 1
                else:
                    for c in gt_cls:
                        class_fn[int(c)] += 1

                list_per_image_pred.append(pred_cnt)
                list_per_image_tp.append(img_tp)
                list_per_image_fp.append(img_fp)

    # 1. Detection (class-agnostic, IoU matching)
    float_det_prec = int_total_matched / int_total_pred if int_total_pred > 0 else 0
    float_det_rec  = int_total_matched / int_total_gt   if int_total_gt > 0 else 0
    float_det_f1   = (2 * float_det_prec * float_det_rec /
                      (float_det_prec + float_det_rec)
                      if (float_det_prec + float_det_rec) > 0 else 0)

    # 2. Matching stats
    float_mean_iou = float(np.mean(list_matched_ious)) if list_matched_ious else 0.0
    float_mle     = float(np.mean(list_matched_distances)) if list_matched_distances else 0.0
    float_mle_std = float(np.std(list_matched_distances))  if list_matched_distances else 0.0

    # 3. Classification (matched only)
    float_cls_acc = int_correct_cls / int_total_matched if int_total_matched > 0 else 0

    prec, rec, f1 = [], [], []
    for c in range(num_cls):
        p = class_tp[c] / (class_tp[c] + class_fp[c]) if (class_tp[c] + class_fp[c]) > 0 else 0
        r = class_tp[c] / (class_tp[c] + class_fn[c]) if (class_tp[c] + class_fn[c]) > 0 else 0
        f = 2*p*r/(p+r) if (p+r) > 0 else 0
        prec.append(p); rec.append(r); f1.append(f)

    total_tp = class_tp.sum()
    total_fp = class_fp.sum()
    total_fn = class_fn.sum()
    micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_rec  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1   = 2*micro_prec*micro_rec/(micro_prec+micro_rec) if (micro_prec+micro_rec) > 0 else 0

    wts = class_gt_count / class_gt_count.sum() if class_gt_count.sum() > 0 else np.zeros(num_cls)
    weighted_f1 = (wts * np.array(f1)).sum()

    # 4. PQ = DQ × CQ
    float_dq = float_det_f1
    float_cq = float_cls_acc
    float_pq = float_dq * float_cq

    # 5. Per-image
    int_n_images = len(list_per_image_gt)
    float_avg_fp = sum(list_per_image_fp) / int_n_images if int_n_images > 0 else 0

    return {
        'det_precision': float_det_prec, 'det_recall': float_det_rec, 'det_f1': float_det_f1,
        'total_gt': int_total_gt, 'total_pred': int_total_pred, 'total_matched': int_total_matched,
        'mean_iou': float_mean_iou,
        'matched_ious': np.array(list_matched_ious),
        'mle': float_mle, 'mle_std': float_mle_std,
        'matched_distances': np.array(list_matched_distances),
        'cls_accuracy': float_cls_acc,
        'class_tp': class_tp, 'class_fp': class_fp, 'class_fn': class_fn,
        'class_gt_count': class_gt_count,
        'precision': np.array(prec), 'recall': np.array(rec), 'f1': np.array(f1),
        'macro_precision': np.mean(prec), 'macro_recall': np.mean(rec), 'macro_f1': np.mean(f1),
        'micro_precision': micro_prec, 'micro_recall': micro_rec, 'micro_f1': micro_f1,
        'weighted_f1': weighted_f1,
        'DQ': float_dq, 'CQ': float_cq, 'PQ': float_pq,
        'confusion': confusion,
        'all_detections': list_all_detections, 'all_gt_counts': np_all_gt_counts,
        'per_image_gt': np.array(list_per_image_gt),
        'per_image_pred': np.array(list_per_image_pred),
        'per_image_tp': np.array(list_per_image_tp),
        'per_image_fp': np.array(list_per_image_fp),
        'n_images': int_n_images, 'avg_fp_per_image': float_avg_fp,
    }


# ── Run evaluation ───────────────────────────────────────────────
CONF_TH       = 0.1
NMS_IOU_TH    = 0.3   # class-agnostic NMS IoU
IOU_MATCH_TH  = 0.3   # GT-Pred 매칭 IoU 임계값

results = full_evaluation(model, val_loader, device, params,
                          conf_threshold=CONF_TH,
                          nms_iou_threshold=NMS_IOU_TH,
                          iou_match_threshold=IOU_MATCH_TH)
print("✅ Evaluation complete.")
print(f"   DQ={results['DQ']:.4f}  CQ={results['CQ']:.4f}  PQ={results['PQ']:.4f}")
print(f"   Mean IoU (matched) = {results['mean_iou']:.4f}")
print(f"   MLE = {results['mle']:.2f} ± {results['mle_std']:.2f} px")


## Table 1 — Per-class Point Detection + Classification Performance

In [ ]:
rows = []
for c in range(num_classes_her2):
    rows.append({
        'Class': class_names_her2[c],
        'GT': int(results['class_gt_count'][c]),
        'TP': int(results['class_tp'][c]),
        'FP': int(results['class_fp'][c]),
        'FN': int(results['class_fn'][c]),
        'Precision': results['precision'][c],
        'Recall': results['recall'][c],
        'F1-score': results['f1'][c],
    })

df_class = pd.DataFrame(rows)

macro_row = {
    'Class': 'Macro Avg',
    'GT': int(results['class_gt_count'].sum()),
    'TP': int(results['class_tp'].sum()),
    'FP': int(results['class_fp'].sum()),
    'FN': int(results['class_fn'].sum()),
    'Precision': results['macro_precision'],
    'Recall': results['macro_recall'],
    'F1-score': results['macro_f1'],
}
micro_row = {
    'Class': 'Micro Avg',
    'GT': '', 'TP': '', 'FP': '', 'FN': '',
    'Precision': results['micro_precision'],
    'Recall': results['micro_recall'],
    'F1-score': results['micro_f1'],
}
weighted_row = {
    'Class': 'Weighted Avg',
    'GT': '', 'TP': '', 'FP': '', 'FN': '',
    'Precision': '', 'Recall': '',
    'F1-score': results['weighted_f1'],
}
df_table1 = pd.concat([df_class, pd.DataFrame([macro_row, micro_row, weighted_row])],
                      ignore_index=True)

df_table1.style.format({
    'Precision': lambda v: f'{v:.4f}' if isinstance(v, (int, float)) and v != '' else '',
    'Recall':    lambda v: f'{v:.4f}' if isinstance(v, (int, float)) and v != '' else '',
    'F1-score':  lambda v: f'{v:.4f}' if isinstance(v, (int, float)) and v != '' else '',
}).set_caption('Table 1. Per-class point detection + classification').hide(axis='index')


## Table 2 — Point Detection Summary (DQ / CQ / PQ / MLE)

In [ ]:
summary_rows = [
    {'Category': 'Detection (class-agnostic)', 'Metric': 'Total GT boxes',       'Value': f"{results['total_gt']}"},
    {'Category': '',                           'Metric': 'Total Predictions',      'Value': f"{results['total_pred']}"},
    {'Category': '',                           'Metric': 'Matched (TP_det)',       'Value': f"{results['total_matched']}"},
    {'Category': '',                           'Metric': 'Detection Precision',    'Value': f"{results['det_precision']:.4f}"},
    {'Category': '',                           'Metric': 'Detection Recall',       'Value': f"{results['det_recall']:.4f}"},
    {'Category': '',                           'Metric': 'Detection F1 (= DQ)',    'Value': f"{results['det_f1']:.4f}"},
    {'Category': 'Localization',               'Metric': 'Mean IoU (matched)',     'Value': f"{results['mean_iou']:.4f}"},
    {'Category': '',                           'Metric': 'Mean Center Err. (px)',  'Value': f"{results['mle']:.2f} ± {results['mle_std']:.2f}"},
    {'Category': '',                           'Metric': 'Median Center Err. (px)','Value': f"{np.median(results['matched_distances']):.2f}" if len(results['matched_distances']) > 0 else "N/A"},
    {'Category': 'Classification (matched)',   'Metric': 'Accuracy (= CQ)',        'Value': f"{results['cls_accuracy']:.4f}"},
    {'Category': '',                           'Metric': 'Macro F1',               'Value': f"{results['macro_f1']:.4f}"},
    {'Category': '',                           'Metric': 'Micro F1',               'Value': f"{results['micro_f1']:.4f}"},
    {'Category': '',                           'Metric': 'Weighted F1',            'Value': f"{results['weighted_f1']:.4f}"},
    {'Category': 'Panoptic Quality',           'Metric': 'DQ (Detection Quality)', 'Value': f"{results['DQ']:.4f}"},
    {'Category': '',                           'Metric': 'CQ (Classification Q.)', 'Value': f"{results['CQ']:.4f}"},
    {'Category': '',                           'Metric': 'PQ = DQ × CQ',          'Value': f"{results['PQ']:.4f}"},
    {'Category': 'FROC',                       'Metric': 'Avg FP / image',         'Value': f"{results['avg_fp_per_image']:.2f}"},
]
df_summary = pd.DataFrame(summary_rows)
df_summary.style.set_caption(
    f'Table 2. Evaluation summary (NMS class-agnostic, IoU match ≥ {IOU_MATCH_TH})'
).hide(axis='index')


## Figure 1 — Normalized Confusion Matrix

In [ ]:
cm = results['confusion']
cm_norm = cm.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm = cm_norm / row_sums

tick_labels = [short_names[i] for i in range(num_classes_her2)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(cm, cmap='Blues', interpolation='nearest')
for r in range(num_classes_her2):
    for c in range(num_classes_her2):
        val = cm[r, c]
        color = 'white' if val > cm.max() * 0.6 else 'black'
        axes[0].text(c, r, f'{val}', ha='center', va='center', fontsize=10, color=color)
axes[0].set_xticks(range(num_classes_her2)); axes[0].set_xticklabels(tick_labels, rotation=45, ha='right')
axes[0].set_yticks(range(num_classes_her2)); axes[0].set_yticklabels(tick_labels)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Ground Truth')
axes[0].set_title('(a) Confusion Matrix (Counts)')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(cm_norm, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
for r in range(num_classes_her2):
    for c in range(num_classes_her2):
        val = cm_norm[r, c]
        color = 'white' if val > 0.6 else 'black'
        axes[1].text(c, r, f'{val:.2f}', ha='center', va='center', fontsize=10, color=color)
axes[1].set_xticks(range(num_classes_her2)); axes[1].set_xticklabels(tick_labels, rotation=45, ha='right')
axes[1].set_yticks(range(num_classes_her2)); axes[1].set_yticklabels(tick_labels)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Ground Truth')
axes[1].set_title('(b) Confusion Matrix (Row-normalized)')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('Figure 1. Confusion Matrix (HER2 IHC)', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'fig1_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 2 — Per-class Precision / Recall / F1 Bar Chart

In [ ]:
x = np.arange(num_classes_her2)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars_p = ax.bar(x - width, results['precision'], width, label='Precision', color='#4C72B0', edgecolor='white')
bars_r = ax.bar(x,         results['recall'],    width, label='Recall',    color='#DD8452', edgecolor='white')
bars_f = ax.bar(x + width, results['f1'],        width, label='F1-score',  color='#55A868', edgecolor='white')

for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([short_names[i] for i in range(num_classes_her2)], rotation=0, ha='center')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Figure 2. Per-class Precision, Recall, and F1-score (HER2)', fontweight='bold')
ax.legend(loc='upper right')
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax.grid(axis='y', alpha=0.3)

ax.axhline(results['macro_precision'], color='#4C72B0', ls='--', lw=0.8, alpha=0.6)
ax.axhline(results['macro_recall'],    color='#DD8452', ls='--', lw=0.8, alpha=0.6)
ax.axhline(results['macro_f1'],        color='#55A868', ls='--', lw=0.8, alpha=0.6)
ax.text(num_classes_her2 - 0.5, results['macro_f1'] + 0.015, f'Macro F1={results["macro_f1"]:.3f}',
        fontsize=9, color='#55A868')

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig2_per_class_bar.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 3 — Precision-Recall Curve per Class

In [ ]:
class_colors = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#808080']

fig, ax = plt.subplots(figsize=(8, 7))

for c in range(num_classes_her2):
    dets = [(conf, tp) for conf, pcls, tp, gc in results['all_detections'] if pcls == c]
    n_gt = results['all_gt_counts'][c]
    if len(dets) == 0 or n_gt == 0:
        ax.plot([0, 1], [0, 0], color=class_colors[c], lw=1.5,
                label=f'{short_names[c]} (AP=0.00)')
        continue

    dets.sort(key=lambda x: -x[0])
    tps = np.array([d[1] for d in dets], dtype=float)
    cum_tp = np.cumsum(tps)
    cum_fp = np.cumsum(1 - tps)
    prec_curve = cum_tp / (cum_tp + cum_fp)
    rec_curve  = cum_tp / n_gt

    mrec = np.concatenate(([0.], rec_curve, [1.]))
    mpre = np.concatenate(([1.], prec_curve, [0.]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])

    ax.plot(rec_curve, prec_curve, color=class_colors[c], lw=1.8,
            label=f'{short_names[c]} (AP={ap:.3f})')

ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Figure 3. Precision-Recall Curve per Class (HER2)', fontweight='bold')
ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig3_pr_curve.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 4 — FROC Curve (Sensitivity vs Avg FP per Image)

In [ ]:
conf_thresholds = np.arange(0.05, 0.96, 0.05)
list_froc_sens, list_froc_fp = [], []
list_sweep_det_f1, list_sweep_pq = [], []
list_sweep_macro_f1, list_sweep_mle = [], []

print("Sweeping confidence thresholds for FROC & metrics...")
for ct in tqdm(conf_thresholds):
    r = full_evaluation(model, val_loader, device, params,
                        conf_threshold=ct, nms_iou_threshold=NMS_IOU_TH,
                        iou_match_threshold=IOU_MATCH_TH)
    list_froc_sens.append(r['det_recall'])
    list_froc_fp.append(r['avg_fp_per_image'])
    list_sweep_det_f1.append(r['det_f1'])
    list_sweep_pq.append(r['PQ'])
    list_sweep_macro_f1.append(r['macro_f1'])
    list_sweep_mle.append(r['mle'])

np_froc_sens    = np.array(list_froc_sens)
np_froc_fp      = np.array(list_froc_fp)
np_sweep_det_f1 = np.array(list_sweep_det_f1)
np_sweep_pq     = np.array(list_sweep_pq)
np_sweep_macro  = np.array(list_sweep_macro_f1)
np_sweep_mle    = np.array(list_sweep_mle)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].plot(np_froc_fp, np_froc_sens, 'o-', color='#c0392b', lw=2.2, markersize=5)
for j, ct in enumerate(conf_thresholds):
    if round(float(ct), 2) in [0.1, 0.2, 0.3, 0.5, 0.7]:
        axes[0].annotate(f'{ct:.1f}', (np_froc_fp[j], np_froc_sens[j]),
                         textcoords='offset points', xytext=(6, 4), fontsize=8, color='gray')
axes[0].set_xlabel('Average FP per Image')
axes[0].set_ylabel('Sensitivity (Detection Recall)')
axes[0].set_title('(a) FROC Curve')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)

axes[1].plot(conf_thresholds, np_sweep_det_f1, 'D-', color='#2980b9', lw=2, markersize=4, label='DQ (Det F1)')
axes[1].plot(conf_thresholds, np_sweep_pq,     's-', color='#27ae60', lw=2, markersize=4, label='PQ')
axes[1].plot(conf_thresholds, np_sweep_macro,   'o-', color='#e67e22', lw=2, markersize=4, label='Macro F1 (cls)')
best_pq_idx = int(np.argmax(np_sweep_pq))
axes[1].axvline(conf_thresholds[best_pq_idx], color='gray', ls='--', lw=1, alpha=0.5)
axes[1].text(conf_thresholds[best_pq_idx] + 0.02, np_sweep_pq[best_pq_idx] - 0.04,
             f'Best PQ={np_sweep_pq[best_pq_idx]:.3f}\n@ conf={conf_thresholds[best_pq_idx]:.2f}',
             fontsize=9, color='#27ae60')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('(b) Point Metrics vs Confidence Threshold')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

fig.suptitle('Figure 4. FROC Curve & Confidence Threshold Sensitivity (HER2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig4_froc_conf_sweep.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ Best PQ       = {np_sweep_pq[best_pq_idx]:.4f} @ conf = {conf_thresholds[best_pq_idx]:.2f}")
best_f1_idx = int(np.argmax(np_sweep_macro))
print(f"   Best Macro F1 = {np_sweep_macro[best_f1_idx]:.4f} @ conf = {conf_thresholds[best_f1_idx]:.2f}")


## Figure 5 — Qualitative Comparison: GT vs Prediction (Multiple Samples)

In [ ]:
colors_vis = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#808080']
num_samples = min(6, len(val_dataset))
sample_indices = random.sample(range(len(val_dataset)), num_samples)

fig, axes = plt.subplots(num_samples, 2, figsize=(14, 4.5 * num_samples))
if num_samples == 1:
    axes = axes[np.newaxis, :]

for row, sidx in enumerate(sample_indices):
    img_t, cls_t, box_t, _ = val_dataset[sidx]
    img_np = (img_t.numpy().transpose(1, 2, 0)).astype(np.uint8)
    h_img, w_img = img_np.shape[:2]

    axes[row, 0].imshow(img_np)
    for j in range(len(cls_t)):
        cid = int(cls_t[j].item())
        cx, cy = box_t[j, 0].item() * w_img, box_t[j, 1].item() * h_img
        axes[row, 0].scatter(cx, cy, facecolors='none', s=25, marker='o',
                             edgecolors=colors_vis[cid], linewidths=1.2)
    axes[row, 0].set_title(f'GT (sample {sidx}, n={len(cls_t)})', fontsize=11)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(img_np)
    with torch.no_grad():
        inp = img_t.unsqueeze(0).to(device).float() / 255.
        with torch.amp.autocast('cuda'):
            pred = model(inp)
        det = util.non_max_suppression(pred, confidence_threshold=CONF_TH, iou_threshold=NMS_IOU_TH, class_agnostic=True)
    n_pred = 0
    if len(det[0]) > 0:
        for *xyxy, conf, cid in det[0]:
            cx = ((xyxy[0] + xyxy[2]) / 2).item()
            cy = ((xyxy[1] + xyxy[3]) / 2).item()
            axes[row, 1].scatter(cx, cy, facecolors='none', s=25, marker='o',
                                 edgecolors=colors_vis[int(cid.item())], linewidths=1.2)
            n_pred += 1
    axes[row, 1].set_title(f'Prediction (n={n_pred})', fontsize=11)
    axes[row, 1].axis('off')

legend_els = [patches.Patch(color=colors_vis[i], label=short_names[i])
              for i in range(num_classes_her2)]
fig.legend(handles=legend_els, loc='lower center', ncol=num_classes_her2,
           bbox_to_anchor=(0.5, -0.01), fontsize=9)
fig.suptitle('Figure 5. Qualitative Comparison — Ground Truth vs Model Prediction (HER2)',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig5_qualitative.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 6 — IoU Match Threshold Sensitivity & Localization Quality


In [ ]:
iou_match_thresholds = np.arange(0.1, 0.76, 0.05)
list_iou_dq, list_iou_pq, list_iou_mean_iou = [], [], []

print("Sweeping IoU match thresholds...")
for it in tqdm(iou_match_thresholds):
    r = full_evaluation(model, val_loader, device, params,
                        conf_threshold=CONF_TH,
                        nms_iou_threshold=NMS_IOU_TH,
                        iou_match_threshold=float(it))
    list_iou_dq.append(r['det_f1'])
    list_iou_pq.append(r['PQ'])
    list_iou_mean_iou.append(r['mean_iou'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) DQ & PQ vs IoU match threshold
axes[0].plot(iou_match_thresholds, list_iou_dq, 'o-', color='#2980b9', lw=2, markersize=5, label='DQ (Det F1)')
axes[0].plot(iou_match_thresholds, list_iou_pq, 's-', color='#27ae60', lw=2, markersize=5, label='PQ')
axes[0].axvline(IOU_MATCH_TH, color='red', ls='--', lw=1.2, alpha=0.6, label=f'Current ({IOU_MATCH_TH:.2f})')
axes[0].set_xlabel('IoU Match Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('(a) DQ & PQ vs IoU Match Threshold')
axes[0].legend(loc='lower left')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)

# (b) Matched IoU distribution
ious = results['matched_ious']
if len(ious) > 0:
    axes[1].hist(ious, bins=30, color='#3498db', edgecolor='white', alpha=0.8, density=True)
    axes[1].axvline(results['mean_iou'], color='red', ls='--', lw=1.5,
                    label=f'Mean={results["mean_iou"]:.3f}')
    axes[1].axvline(np.median(ious), color='orange', ls='--', lw=1.5,
                    label=f'Median={np.median(ious):.3f}')
    axes[1].axvline(IOU_MATCH_TH, color='gray', ls=':', lw=1.2,
                    label=f'Threshold={IOU_MATCH_TH:.2f}')
    axes[1].set_xlabel('Matched IoU')
    axes[1].set_ylabel('Density')
    axes[1].set_title(f'(b) Matched IoU Distribution (n={len(ious)})')
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

# (c) Center localization error CDF
dists = results['matched_distances']
if len(dists) > 0:
    sorted_d = np.sort(dists)
    cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    axes[2].plot(sorted_d, cdf, color='#2c3e50', lw=2)
    axes[2].axhline(0.9, color='gray', ls=':', lw=1, alpha=0.5)
    pct90 = np.percentile(dists, 90)
    axes[2].axvline(pct90, color='orange', ls='--', lw=1.2,
                    label=f'90th pct = {pct90:.1f}px')
    axes[2].set_xlabel('Center Localization Error (px)')
    axes[2].set_ylabel('Cumulative Proportion')
    axes[2].set_title('(c) CDF of Center Localization Error')
    axes[2].legend(fontsize=9)
    axes[2].grid(alpha=0.3)
    axes[2].set_ylim(0, 1.02)

fig.suptitle('Figure 6. IoU Match Threshold Sensitivity & Localization Quality (HER2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig6_iou_threshold_localization.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 7 — Error Analysis: FP / FN Distribution

In [ ]:
x = np.arange(num_classes_her2)
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(x, results['class_tp'], width, label='TP', color='#55A868')
axes[0].bar(x, results['class_fn'], width, bottom=results['class_tp'],
            label='FN (Missed)', color='#C44E52')
for i in x:
    total = results['class_tp'][i] + results['class_fn'][i]
    if total > 0:
        axes[0].text(i, total + 5, f'{results["recall"][i]:.2f}',
                     ha='center', fontsize=8, color='#333')
axes[0].set_xticks(x)
axes[0].set_xticklabels([short_names[i] for i in range(num_classes_her2)], rotation=0, ha='center')
axes[0].set_ylabel('Count')
axes[0].set_title('(a) TP vs FN per Class (Recall view)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x, results['class_tp'], width, label='TP', color='#55A868')
axes[1].bar(x, results['class_fp'], width, bottom=results['class_tp'],
            label='FP (False Alarm)', color='#4C72B0')
for i in x:
    total = results['class_tp'][i] + results['class_fp'][i]
    if total > 0:
        axes[1].text(i, total + 5, f'{results["precision"][i]:.2f}',
                     ha='center', fontsize=8, color='#333')
axes[1].set_xticks(x)
axes[1].set_xticklabels([short_names[i] for i in range(num_classes_her2)], rotation=0, ha='center')
axes[1].set_ylabel('Count')
axes[1].set_title('(b) TP vs FP per Class (Precision view)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Figure 7. Error Analysis — TP / FP / FN Distribution (HER2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig7_error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()


## Figure 8 — Per-image GT Count vs Predicted Count Scatter

In [ ]:
from scipy.stats import pearsonr

gt_total   = results['per_image_gt'].sum(axis=1)
pred_total = results['per_image_pred'].sum(axis=1)
r_val, p_val = pearsonr(gt_total, pred_total)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

max_val = max(gt_total.max(), pred_total.max()) * 1.1
axes[0].scatter(gt_total, pred_total, alpha=0.5, s=30, edgecolors='k', linewidths=0.3)
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1.2, label='y = x (ideal)')
axes[0].set_xlabel('GT cell count per image')
axes[0].set_ylabel('Predicted cell count per image')
axes[0].set_title(f'(a) Total Cell Count (r={r_val:.3f}, p={p_val:.2e})')
axes[0].legend(loc='upper left')
axes[0].set_xlim(0, max_val); axes[0].set_ylim(0, max_val)
axes[0].set_aspect('equal')
axes[0].grid(alpha=0.3)

for c in range(num_classes_her2):
    gt_c = results['per_image_gt'][:, c]
    pred_c = results['per_image_pred'][:, c]
    mask = (gt_c > 0) | (pred_c > 0)
    if mask.sum() == 0:
        continue
    axes[1].scatter(gt_c[mask], pred_c[mask], alpha=0.4, s=20,
                    color=class_colors[c], label=short_names[c])

max_c = max(results['per_image_gt'].max(), results['per_image_pred'].max()) * 1.1
axes[1].plot([0, max_c], [0, max_c], 'r--', lw=1.2)
axes[1].set_xlabel('GT count per image')
axes[1].set_ylabel('Predicted count per image')
axes[1].set_title('(b) Per-class Count Correlation')
axes[1].legend(loc='upper left', fontsize=8, markerscale=1.5)
axes[1].set_xlim(0, max_c); axes[1].set_ylim(0, max_c)
axes[1].set_aspect('equal')
axes[1].grid(alpha=0.3)

fig.suptitle('Figure 8. GT vs Predicted Cell Count per Image (HER2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'fig8_count_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()


## Export Results to CSV (for LaTeX / supplementary)

In [ ]:
csv_path_1 = os.path.join(save_dir, 'table1_per_class_metrics.csv')
csv_path_2 = os.path.join(save_dir, 'table2_point_detection_summary.csv')

df_table1.to_csv(csv_path_1, index=False)
df_summary.to_csv(csv_path_2, index=False)

# LaTeX Table 1
print("=" * 70)
print("LaTeX Table 1:")
print("=" * 70)
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Per-class point detection and classification performance (HER2)")
print(f"  (confidence $\\geq$ {CONF_TH}, IoU match $\\geq$ {IOU_MATCH_TH:.2f}).")
print(r"\label{tab:her2_per_class}")
print(r"\begin{tabular}{lrrrrrrr}")
print(r"\toprule")
print(r"Class & GT & TP & FP & FN & Precision & Recall & F1 \\")
print(r"\midrule")
for _, row in df_table1.iterrows():
    name = row['Class']
    if name in ('Macro Avg', 'Micro Avg', 'Weighted Avg'):
        print(r"\midrule")
    gt = int(row['GT']) if row['GT'] != '' else '--'
    tp = int(row['TP']) if row['TP'] != '' else '--'
    fp = int(row['FP']) if row['FP'] != '' else '--'
    fn = int(row['FN']) if row['FN'] != '' else '--'
    p  = f"{row['Precision']:.4f}" if isinstance(row['Precision'], (int, float)) and row['Precision'] != '' else '--'
    r_ = f"{row['Recall']:.4f}"    if isinstance(row['Recall'], (int, float)) and row['Recall'] != '' else '--'
    f1 = f"{row['F1-score']:.4f}"  if isinstance(row['F1-score'], (int, float)) and row['F1-score'] != '' else '--'
    print(f"{name} & {gt} & {tp} & {fp} & {fn} & {p} & {r_} & {f1} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# LaTeX Table 2
print("\n" + "=" * 70)
print("LaTeX Table 2 (Point Detection Summary):")
print("=" * 70)
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Point-detection evaluation summary (HER2).}")
print(r"\label{tab:her2_summary}")
print(r"\begin{tabular}{llr}")
print(r"\toprule")
print(r"Category & Metric & Value \\")
print(r"\midrule")
for _, row in df_summary.iterrows():
    cat = row['Category'] if row['Category'] else ''
    print(f"{cat} & {row['Metric']} & {row['Value']} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

print(f"\n✅ CSV 저장: {csv_path_1}")
print(f"✅ CSV 저장: {csv_path_2}")
print(f"✅ 모든 figure 저장 위치: {save_dir}")


In [ ]:
# 수치 확인용
print("=== 설정 ===")
print(f"CONF_TH={CONF_TH}, NMS_IOU_TH={NMS_IOU_TH}, IOU_MATCH_TH={IOU_MATCH_TH}")
print(f"NMS: class-agnostic (모든 클래스 통합)")
print(f"Val images: {len(val_dataset)}")
print(f"Epoch = {checkpoint.get('epoch', 'N/A')}")

print("\n=== Table 2 수치 ===")
print(f"Total GT      : {results['total_gt']}")
print(f"Total Pred    : {results['total_pred']}")
print(f"Matched       : {results['total_matched']}")
print(f"Det Precision : {results['det_precision']:.4f}")
print(f"Det Recall    : {results['det_recall']:.4f}")
print(f"DQ (Det F1)   : {results['DQ']:.4f}")
print(f"Mean IoU      : {results['mean_iou']:.4f}")
print(f"MLE (center)  : {results['mle']:.2f} ± {results['mle_std']:.2f}")
print(f"Median LE     : {np.median(results['matched_distances']):.2f}")
print(f"CQ (cls acc)  : {results['CQ']:.4f}")
print(f"Macro F1      : {results['macro_f1']:.4f}")
print(f"Micro F1      : {results['micro_f1']:.4f}")
print(f"Weighted F1   : {results['weighted_f1']:.4f}")
print(f"PQ            : {results['PQ']:.4f}")
print(f"Avg FP/img    : {results['avg_fp_per_image']:.2f}")

print("\n=== Table 1 수치 ===")
for c in range(num_classes_her2):
    print(f"{short_names[c]:6s}: GT={int(results['class_gt_count'][c]):5d}, "
          f"TP={int(results['class_tp'][c]):5d}, "
          f"FP={int(results['class_fp'][c]):5d}, "
          f"FN={int(results['class_fn'][c]):5d}, "
          f"P={results['precision'][c]:.4f}, "
          f"R={results['recall'][c]:.4f}, "
          f"F1={results['f1'][c]:.4f}")
print(f"\nMacro: P={results['macro_precision']:.4f}, R={results['macro_recall']:.4f}, F1={results['macro_f1']:.4f}")
print(f"Micro: P={results['micro_precision']:.4f}, R={results['micro_recall']:.4f}, F1={results['micro_f1']:.4f}")
